In [1]:
# importing necessary libraries
import numpy as np
import pandas as pd
import os
import glob

# import matplotlib.pyplot as plt
# import seaborn as sns

In [2]:
#Read the CSV from the data folder
df = pd.read_csv("data/raw/cardiac_complications.csv")

In [3]:
# ============================================================
# STEP 1: RENAME COLUMNS
#
# Descriptive column names that aren't too long
# ============================================================
df = df.rename(columns={
    "inpatient_number": "patient_id",
    "nyha_cardiac_function_classification": "nyha_class",
    "killip_grade": "killip_grade",
    "myocardial_infarction": "heart_attack",
    "congestive_heart_failure": "congestive_hf",
    "peripheral_vascular_disease": "peripheral_vasc_disease",
    "type_of_heart_failure": "hf_type",
    "lvef": "ejection_fraction",
    "left_ventricular_end_diastolic_diameter_lv": "lv_diameter",
    "mitral_valve_ems": "mitral_e_velocity",
    "mitral_valve_ams": "mitral_a_velocity",
    "ea": "e_a_ratio",
    "tricuspid_valve_return_velocity": "tricuspid_velocity",
    "tricuspid_valve_return_pressure": "tricuspid_pressure",
})

print(df.shape)

(2008, 14)


In [4]:
# ============================================================
# STEP 2: REMOVE DUPLICATE ROWS
# ============================================================
duplicate_count = df.duplicated().sum()
print("\nDuplicate rows found:", duplicate_count)
df = df.drop_duplicates().copy()

duplicate_ids = df["patient_id"].duplicated().sum()
print("Repeated patient IDs found:", duplicate_ids)
if duplicate_ids > 0:
    print(df[df["patient_id"].duplicated(keep=False)].sort_values("patient_id"))



Duplicate rows found: 0
Repeated patient IDs found: 0


In [5]:
# ============================================================
# STEP 3: CLEAN TEXT VALUES
# This step cleans up the text in the hf_type column, which holds the type of heart failure
# ============================================================
df["hf_type"] = df["hf_type"].astype("string").str.strip().str.title()
print("\nhf_type values:", df["hf_type"].value_counts().to_dict())


hf_type values: {'Both': 1480, 'Left': 477, 'Right': 51}


In [6]:
# ============================================================
# STEP 4: CLEAN CATEGORY COLUMNS
# This step checks 5 columns that should only contain certain numbers, and removes anything that doesn't belong. 
# NYHA class: 1-4, Killip grade: 1-4, disease columns: 0 = No, 1 = Yes
# Anything else becomes missing.
# ============================================================
category_rules = {
    "nyha_class": [1, 2, 3, 4],
    "killip_grade": [1, 2, 3, 4],
    "heart_attack": [0, 1],
    "congestive_hf": [0, 1],
    "peripheral_vasc_disease": [0, 1],
}

print()
for col, valid_values in category_rules.items():
    invalid = ~df[col].isin(valid_values)
    print(f"{col} invalid values: {invalid.sum()}")
    if invalid.any():
        df.loc[invalid, col] = np.nan
    df[col] = df[col].astype("Int64")   # keeps whole numbers even if blanks exist


nyha_class invalid values: 0
killip_grade invalid values: 0
heart_attack invalid values: 0
congestive_hf invalid values: 0
peripheral_vasc_disease invalid values: 0


**step-5 cleans the ejection fraction column (LVEF), the percentage of blood the heart pumps out with each beat.
A normal LVEF is 55–70%, but this is a heart failure dataset, so low values like 30% are real patient data, not mistakes. We only remove values that are physically impossible: 0 or below (the heart pumps something), or above 100 (you can't pump out more than 100% of the blood).**

In [7]:
# ============================================================
# STEP 5: CLEAN LVEF (%)
#
# Normal range 55-70%, but low LVEF is common in heart failure,
# so only impossible values (<= 0 or > 100) are removed.
# ============================================================
invalid_lvef = (df["ejection_fraction"] <= 0) | (df["ejection_fraction"] > 100)
print("\nInvalid LVEF values replaced with NaN:", invalid_lvef.sum())
df.loc[invalid_lvef, "ejection_fraction"] = np.nan

# True = normal (55-70), False = outside normal, blank = no value
df["lvef_normal"] = (
    df["ejection_fraction"].between(55, 70)
    .astype("boolean")
    .mask(df["ejection_fraction"].isna())
)


Invalid LVEF values replaced with NaN: 0


**Step-6 cleans the LV diameter column: how wide the heart's main pumping chamber is when it is fully filled with blood.
The comment: the unit problem
The documentation says the unit is cm, with a normal range of 3.5–5.6 cm. But almost every value in the data is between 20 and 100. 
A heart chamber cant not be 53 cm wide, which is bigger than a whole chest. It does make sense as mm, though: 53 mm = 5.3 cm, which is normal. 
So the data was recorded in mm, and we convert it to match the documentation.**

In [8]:
# ============================================================
# STEP 6: CLEAN LV END-DIASTOLIC DIAMETER (cm)
#
# Documentation says cm (normal 3.5-5.6), but almost every value
# is 20-100, which only makes sense as mm -> divide by 10.
# After conversion, anything under 2 cm or over 10 cm is not a
# realistic adult LV diameter -> missing.
# ============================================================
lv_mm_values = df["lv_diameter"].between(20, 100)
print("\nLV diameter values converted mm -> cm:", lv_mm_values.sum())
df.loc[lv_mm_values, "lv_diameter"] = df.loc[lv_mm_values, "lv_diameter"] / 10

invalid_lv = (df["lv_diameter"] < 2.0) | (df["lv_diameter"] > 10.0)
print("Invalid LV diameter values replaced with NaN:", invalid_lv.sum())
df.loc[invalid_lv, "lv_diameter"] = np.nan


LV diameter values converted mm -> cm: 1309
Invalid LV diameter values replaced with NaN: 2


**Step 7 Reasoning:
Mitral E and A velocities are recorded in m/s, where real values in this dataset go up to about 3.9. Values above 5 can't be in m/s. Only one row (patient 728235, E = 44, A = 99) could be proven to be in cm/s, because 44 ÷ 99 = 0.44 matches its recorded E/A ratio, so it was converted to 0.44 and 0.99 m/s. The other values above 5 (including 401–409) couldn't be verified, so they were set to missing instead of guessed. Every changed row is marked in mitral_was_fixed (17 rows).**

In [9]:
# ============================================================
# STEP 7: FIX MITRAL VELOCITY VALUES (m/s)
#
# Real values run up to about 3.9 m/s, so that range is kept.
# ============================================================
mitral_cols = ["mitral_e_velocity", "mitral_a_velocity"]
e = df["mitral_e_velocity"]
a = df["mitral_a_velocity"]

# exception: a row where E and A are both in cm/s AND the ratio column proves it
# (patient 728235: 44 / 99 = 0.44, which matches its e_a_ratio)
proven_cm = (e > 5) & (a > 5) & ((e / a - df["e_a_ratio"]).abs() < 0.05)
df.loc[proven_cm, mitral_cols] = df.loc[proven_cm, mitral_cols] / 100
print("\nMitral rows converted from cm/s (proven by ratio):", proven_cm.sum())

# everything else above 5 is invalid -> set to missing
for col in mitral_cols:
    invalid = df[col] > 5
    print(f"{col}: {invalid.sum()} set to missing")
    df.loc[invalid, col] = np.nan


Mitral rows converted from cm/s (proven by ratio): 1
mitral_e_velocity: 10 set to missing
mitral_a_velocity: 6 set to missing


**Step-8
Missing E/A ratios were calculated as E ÷ A wherever both values existed (140 rows, marked in ea_was_filled), since the recorded ratio matches E ÷ A in 97% of rows. Abnormal values were flagged instead of removed, because they're expected in heart failure patients.**

In [10]:
# ============================================================
# STEP 8: FILL THE E/A RATIO
#
# The ratio column matches E / A in about 97% of rows that have
# all three, so missing ratios are calculated from E and A.
# ============================================================
calc_ratio = (df["mitral_e_velocity"] / df["mitral_a_velocity"]).round(3)

can_fill = df["e_a_ratio"].isna() & calc_ratio.notna()
df.loc[can_fill, "e_a_ratio"] = calc_ratio[can_fill]
print("\ne_a_ratio filled from E / A:", can_fill.sum())



e_a_ratio filled from E / A: 140


**High tricuspid velocities can be real in severe heart or lung disease, so they were kept. Only zero or negative values were treated as invalid, since blood flow speed can't be zero or below. No invalid values were found (range 0.9–5.76 m/s).**

In [11]:

# ============================================================
# STEP 9: CLEAN TRICUSPID RETURN VELOCITY (m/s)
#
# High values can occur in severe cardiac/pulmonary disease,
# so they are kept. Only zero or negative values are removed.
# ============================================================
invalid_tr_velocity = df["tricuspid_velocity"] <= 0
print("\nInvalid tricuspid velocity values:", invalid_tr_velocity.sum())
df.loc[invalid_tr_velocity, "tricuspid_velocity"] = np.nan



Invalid tricuspid velocity values: 0


In [12]:
# ============================================================
# STEP 10: CLEAN TRICUSPID RETURN PRESSURE (mmHg)
#
# High pressures can occur in disease, so they are kept.
# Only zero or negative values are removed.
# ============================================================
invalid_tr_pressure = df["tricuspid_pressure"] <= 0
print("Invalid tricuspid pressure values:", invalid_tr_pressure.sum())
df.loc[invalid_tr_pressure, "tricuspid_pressure"] = np.nan

Invalid tricuspid pressure values: 0


In [13]:
# ============================================================
# STEP 11: ROUND CLEANED MEDICAL VALUES
# ============================================================
measurement_cols = [
    "ejection_fraction",
    "lv_diameter",
    "mitral_e_velocity",
    "mitral_a_velocity",
    "e_a_ratio",
    "tricuspid_velocity",
    "tricuspid_pressure",
]
df[measurement_cols] = df[measurement_cols].round(3)

In [14]:
# ============================================================
# STEP 12: MISSING VALUE REPORT
# ============================================================
missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2),
}).sort_values("missing_percent", ascending=False)

print("\nMissing Values After Cleaning:")
print(missing_report)


Missing Values After Cleaning:
                         missing_count  missing_percent
tricuspid_pressure                1826            90.94
e_a_ratio                         1475            73.46
mitral_a_velocity                 1464            72.91
ejection_fraction                 1373            68.38
lvef_normal                       1373            68.38
tricuspid_velocity                1218            60.66
mitral_e_velocity                 1038            51.69
lv_diameter                        699            34.81
patient_id                           0             0.00
nyha_class                           0             0.00
killip_grade                         0             0.00
heart_attack                         0             0.00
congestive_hf                        0             0.00
peripheral_vasc_disease              0             0.00
hf_type                              0             0.00


In [15]:
# ============================================================
# STEP 13: DISPLAY CLEANED MEDICAL DATA
# ============================================================
print("\nCleaned medical values:")
print(df[measurement_cols].describe().round(2))


Cleaned medical values:
       ejection_fraction  lv_diameter  mitral_e_velocity  mitral_a_velocity  \
count             635.00      1309.00             970.00             544.00   
mean               50.68         5.32               1.06               0.83   
std                13.22         1.07               0.46               0.32   
min                 5.00         2.20               0.03               0.06   
25%                41.00         4.50               0.76               0.58   
50%                51.00         5.30               1.00               0.82   
75%                61.00         6.00               1.26               1.04   
max                82.00         8.80               3.90               2.80   

       e_a_ratio  tricuspid_velocity  tricuspid_pressure  
count     533.00              790.00              182.00  
mean        1.29                2.99               35.91  
std         1.18                0.62               13.70  
min         0.06           

In [16]:
# ============================================================
# STEP 14: RESET INDEX + SAVE
# ============================================================
df = df.reset_index(drop=True)
df.to_csv("data/cleaned/cardiac_complications_cleaned.csv", index=False)

print("\nCleaning completed successfully.")
print("Final dataset shape:", df.shape)


Cleaning completed successfully.
Final dataset shape: (2008, 15)
